# LifeMate — lm-A-058: Parakeet CTC tiếng Việt chưa train, giải mã toàn bộ train và public

**Import → Save Version → Save & Run All**, T4 x2, Internet On. Input duy nhất: LifeMate dataset (version 1).

**Vì sao:** trộn Whisper + Gipformer tăng LB từ 57.12 lên 60.36. Parakeet có kiến trúc khác hẳn nên có thể sai ở chỗ khác, tạo được ASR thứ ba để bỏ phiếu. `nvidia/parakeet-ctc-0.6b-Vietnamese`, 0.6B tham số theo model card, phát hành 01/2026, NVIDIA Open Model License; tổng hệ thống khoảng 5.63B.

**Làm gì:** cài NeMo trong venv Python 3.12 riêng → giải mã 5036 audio train và 1933 audio public, không train. Model chưa học audio train của cuộc thi nên WER trên train là phép đo công bằng, và transcript train dùng được khi train lại LLM. Ước tính ~40 phút.

Output: `parakeet/asr_train_parakeet.shard0.jsonl`, `parakeet/asr_public_test_parakeet.shard0.jsonl`, `parakeet/REPORT.json`.


In [ ]:
MODEL = 'nvidia/parakeet-ctc-0.6b-Vietnamese'
REVISION = 'b0493142b49458810324e3db8be9e8e07b4ebc17'  # commit 2026-02-07, trước hạn 26/10/2026 của đề
NEMO_FILE = 'parakeet-ctc-0.6b-vi.nemo'
NEMO_SPEC = 'nemo_toolkit[asr]==2.6.*'
BATCH_SIZE = 32
EXPECT_AUDIO = {'train': 5036, 'public_test': 1933}


In [ ]:
import glob, json, os, re, shutil, subprocess, sys, time
from pathlib import Path
WORK = Path('/kaggle/working'); OUT = WORK / 'parakeet'; OUT.mkdir(exist_ok=True)
if len(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip().splitlines()) != 2:
    raise RuntimeError('Chọn T4 x2')
def run(args, timeout=2400):
    r = subprocess.run(list(map(str, args)), capture_output=True, text=True, timeout=timeout)
    print('$', ' '.join(map(str, args))[:150], '->', r.returncode, flush=True)
    if r.returncode:
        raise RuntimeError((r.stdout + r.stderr)[-3000:])
    return r.stdout
# NeMo ở venv riêng: không đụng torch của môi trường gốc; Python 3.12 vì Kaggle dùng 3.13
VENV = Path('/tmp/venv_nemo'); PY = VENV / 'bin/python'
run([sys.executable, '-m', 'pip', 'install', '-q', 'uv', 'huggingface_hub'])
run([sys.executable, '-m', 'uv', 'venv', '-q', VENV, '--python', '3.12'])
install = [sys.executable, '-m', 'uv', 'pip', 'install', '-q', '--python', PY]
run(install + ['torch==2.7.1', 'torchaudio==2.7.1', '--index-url', 'https://download.pytorch.org/whl/cu126'])
run(install + [NEMO_SPEC])
print(run([PY, '-c', "import torch, nemo, nemo.collections.asr; assert torch.cuda.is_available(); print('NEMO_OK', nemo.__version__, torch.__version__)"])[-200:])
from huggingface_hub import hf_hub_download
NEMO_PATH = hf_hub_download(MODEL, NEMO_FILE, revision=REVISION)
print('model', NEMO_PATH, os.path.getsize(NEMO_PATH) / 1e9, 'GB')


In [ ]:
%%writefile /kaggle/working/asr_fuse.py
"""lm-A-054: word-level fusion of two ASR hypotheses with a count-based language model.

No neural parameters: trigram counts from training text only, plus the words the agent already has for this sample
(profile, assistant turns, tool results). The primary hypothesis keeps its own formatting unless a word really differs.
"""
from __future__ import annotations

import difflib
import math
import re
import unicodedata
from collections import Counter

DIGITS = "không một hai ba bốn năm sáu bảy tám chín".split()
UNITS = [(10**9, "tỷ"), (10**6, "triệu"), (1000, "nghìn")]


def _below_thousand(n: int, full: bool) -> list[str]:
    out = []
    hundreds, rest = n // 100, n % 100
    if hundreds or full:
        out += [DIGITS[hundreds], "trăm"]
    tens, ones = rest // 10, rest % 10
    if tens == 0:
        if ones:
            out += (["linh"] if out else []) + [DIGITS[ones]]
    elif tens == 1:
        out.append("mười")
        if ones:
            out.append("lăm" if ones == 5 else DIGITS[ones])
    else:
        out += [DIGITS[tens], "mươi"]
        if ones:
            out.append({1: "mốt", 4: "tư", 5: "lăm"}.get(ones, DIGITS[ones]))
    return out


def number_words(digits: str) -> list[str]:
    """How a digit string is read aloud. Leading zeros and long strings (phone numbers) are read digit by digit."""
    if (digits.startswith("0") and len(digits) > 1) or len(digits) > 12:
        return [DIGITS[int(d)] for d in digits]
    n = int(digits)
    if n == 0:
        return ["không"]
    out = []
    for value, name in UNITS:
        if n >= value:
            head = n // value
            out += (number_words(str(head)) if head >= 1000 else _below_thousand(head, False)) + [name]
            n %= value
    if n:
        out += _below_thousand(n, bool(out))
    return out


def spoken(token: str) -> list[str]:
    """One raw token as spoken words: lower case, no punctuation, digits read out. '100.000' is one number."""
    token = unicodedata.normalize("NFC", token).lower()
    token = re.sub(r"(?<=\d)[.,](?=\d{3}(?:\D|$))", "", token)
    out = []
    for part in re.findall(r"\w+", token):
        if part.isdigit():
            out += number_words(part)
        else:
            out.append(part)
    return out


def spoken_text(text: str) -> list[str]:
    return [w for token in text.split() for w in spoken(token)]


class TrigramLM:
    def __init__(self, texts):
        self.uni, self.bi, self.tri = Counter(), Counter(), Counter()
        for text in texts:
            w = ["<s>", "<s>"] + spoken_text(text) + ["</s>"]
            self.uni.update(w)
            self.bi.update(zip(w, w[1:]))
            self.tri.update(zip(w, w[1:], w[2:]))
        self.total, self.vocab = sum(self.uni.values()), len(self.uni)

    def logp(self, a: str, b: str, c: str) -> float:
        if self.tri[(a, b, c)]:
            return math.log(self.tri[(a, b, c)] / self.bi[(a, b)])
        if self.bi[(b, c)]:
            return math.log(.4) + math.log(self.bi[(b, c)] / self.uni[b])
        return 2 * math.log(.4) + math.log((self.uni[c] + .5) / (self.total + .5 * self.vocab + .5))

    def score(self, left: list[str], words: list[str], right: list[str]) -> float:
        seq = left[-2:] + words + right[:2]
        return sum(self.logp(seq[i - 2], seq[i - 1], seq[i]) for i in range(2, len(seq)))


def _match_case(word: str, model: str) -> str:
    return word.capitalize() if model[:1].isupper() else word


def fuse(primary: str, secondary: str, lm: TrigramLM, context: set[str], bias: float = -2., context_weight: float = 2.,
         length_penalty: float = 1.) -> str:
    """Primary text with only the disagreeing spans replaced when the language model prefers the secondary words."""
    tokens = primary.split()
    flat, owner = [], []
    for i, token in enumerate(tokens):
        words = spoken(token)
        flat += words
        owner += [i] * len(words)
    other = spoken_text(secondary)
    if not flat or not other:
        return primary
    ops = difflib.SequenceMatcher(None, flat, other, autojunk=False).get_opcodes()
    out, left = [], ["<s>", "<s>"]
    done = 0  # next primary token not yet emitted
    for op, i, j, u, v in ops:
        if op == "equal":
            for t in range(done, owner[j - 1] + 1):
                out.append(tokens[t])
            done = max(done, owner[j - 1] + 1)
            left += flat[i:j]
            continue
        a, b = flat[i:j], other[u:v]
        # A span cutting through one primary token ("118" read as several words) cannot be replaced cleanly.
        inside = 0 < i < len(flat) and owner[i - 1] == owner[i] or (a and j < len(flat) and owner[j - 1] == owner[j])
        right = flat[j:j + 2] + (["</s>"] if j + 2 > len(flat) else [])
        def total(words):
            return lm.score(left, words, right) + context_weight * sum(w in context for w in words) - length_penalty * len(words)
        if inside or total(a) + bias >= total(b):
            if a:
                for t in range(done, owner[j - 1] + 1):
                    out.append(tokens[t])
                done = max(done, owner[j - 1] + 1)
            left += a
            continue
        model = tokens[owner[i]] if a else ""
        if a:
            for t in range(done, owner[i]):
                out.append(tokens[t])
        out += [_match_case(w, model if k == 0 else "") for k, w in enumerate(b)]
        if a:
            done = max(done, owner[j - 1] + 1)
        left += b
    out += tokens[done:]
    return " ".join(out)


In [ ]:
%%writefile /kaggle/working/parakeet_worker.py
"""One GPU: greedy CTC over its share of the audio, resumable by file."""
import json, sys, time
from pathlib import Path
import nemo.collections.asr as nemo_asr
import torch

cfg = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))
shard = int(sys.argv[2])
items = cfg["items"][shard::2]
t0 = time.time()
model = nemo_asr.models.ASRModel.restore_from(cfg["nemo"], map_location=torch.device("cuda")).eval()
load_s = time.time() - t0
print("params", sum(p.numel() for p in model.parameters()), "load", load_s, flush=True)
hyp, t0 = {}, time.time()
for b in range(0, len(items), 512):
    part = items[b:b + 512]
    outs = model.transcribe([it["path"] for it in part], batch_size=cfg["batch_size"], verbose=False)
    outs = outs[0] if isinstance(outs, tuple) else outs
    if len(outs) != len(part):
        raise RuntimeError(f"{len(outs)} outputs for {len(part)} audio")
    for it, o in zip(part, outs):
        hyp[it["key"]] = (o.text if hasattr(o, "text") else str(o)).strip()
    print(f"[shard {shard}] {b + len(part)}/{len(items)} {(time.time() - t0) / (b + len(part)):.3f} s/audio | {hyp[part[-1]['key']][:90]}", flush=True)
Path(cfg["out"], f"hyp.shard{shard}.json").write_text(json.dumps({"s_per_audio": (time.time() - t0) / max(len(items), 1), "model_load_s": load_s,
    "peak_vram_mb": torch.cuda.max_memory_allocated() / 2**20, "n_params": sum(p.numel() for p in model.parameters()), "hyp": hyp}, ensure_ascii=False), encoding="utf-8")


In [ ]:
# Giải mã, ghi cache đúng định dạng các notebook khác đọc, đo WER trên train
def rows_of(split):
    path = Path(sorted(glob.glob(f'/kaggle/input/**/{split}/{split}.jsonl', recursive=True))[0])
    return path.parent, [json.loads(l) for l in path.read_text(encoding='utf-8').splitlines() if l.strip()]
audio_of = lambda r: [r['audio']] + [t['audio'] for t in r.get('history', []) if t.get('role') == 'user']
items, by_split = [], {}
for split in EXPECT_AUDIO:
    root, rows = rows_of(split)
    names = sorted({a for r in rows for a in audio_of(r)})
    if len(names) != EXPECT_AUDIO[split]:
        raise RuntimeError(f'{split}: {len(names)} audio')
    by_split[split] = (rows, names)
    items += [{'key': f'{split}|{a}', 'path': str(root / a)} for a in names]
(OUT / 'config.json').write_text(json.dumps({'nemo': NEMO_PATH, 'items': items, 'batch_size': BATCH_SIZE, 'out': str(OUT)}), encoding='utf-8')
logs = [open(OUT / f'worker{g}.log', 'w') for g in (0, 1)]
procs = [subprocess.Popen([str(PY), str(WORK / 'parakeet_worker.py'), str(OUT / 'config.json'), str(g)], env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(g), PYTHONUNBUFFERED='1'), stdout=logs[g], stderr=subprocess.STDOUT) for g in (0, 1)]
while any(p.poll() is None for p in procs):
    time.sleep(60)
    for g in (0, 1):
        lines = [l for l in (OUT / f'worker{g}.log').read_text(errors='replace').splitlines() if l.startswith('[shard') or l.startswith('params')]
        if lines: print(lines[-1][:170], flush=True)
for g, p in enumerate(procs):
    print((OUT / f'worker{g}.log').read_text(errors='replace')[-1500:])
    if p.returncode: raise RuntimeError(f'Worker {g} lỗi {p.returncode}')
recs = [json.loads((OUT / f'hyp.shard{g}.json').read_text(encoding='utf-8')) for g in (0, 1)]
hyp = {k: v for r in recs for k, v in r['hyp'].items()}
if set(hyp) != {it['key'] for it in items}:
    raise RuntimeError('Thiếu audio')
for split, (rows, names) in by_split.items():
    records = [{'_header': {'run_hash': f'lm-A-058-parakeet-{REVISION[:12]}'}}] + [{'id': a, 'status': 'completed', 'text': hyp[f'{split}|{a}']} for a in names]
    (OUT / f'asr_{split}_parakeet.shard0.jsonl').write_text(''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in records), encoding='utf-8')
sys.path.insert(0, str(WORK))
from asr_fuse import spoken_text
def ed(a, b):
    d = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        p, d[0] = d[0], i
        for j, y in enumerate(b, 1):
            p, d[j] = d[j], min(d[j] + 1, d[j - 1] + 1, p + (x != y))
    return d[-1]
rows = by_split['train'][0]
final = {r['audio']: r['audio_text'] for r in rows}
errs = words = 0
for a, text in final.items():
    g = spoken_text(text); errs += ed(g, spoken_text(hyp[f'train|{a}'])); words += len(g)
report = {'wer_train_final': errs / words, 'n_final': len(final), 'empty': sum(not t for t in hyp.values()), 'n_params': recs[0]['n_params'],
          's_per_audio': [r['s_per_audio'] for r in recs], 'peak_vram_mb': [r['peak_vram_mb'] for r in recs], 'load_s': [r['model_load_s'] for r in recs]}
(OUT / 'REPORT.json').write_text(json.dumps(report, indent=1))
print('REPORT', report)
for a in list(final)[:12]:
    print('G', final[a][:110], '\nH', hyp[f'train|{a}'][:110], '\n')
(OUT / 'config.json').unlink(); [p.unlink() for p in OUT.glob('hyp.shard*.json')]
